# AutoSchemaKG + HotpotQA + local Qwen3.5-2B

This notebook streams a small HotpotQA slice, builds a knowledge graph with a local Qwen model, and packages the graph plus exact provenance. Gold answers are retained for later evaluation but are never sent to KG extraction.

In [ ]:
import os, subprocess, sys
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode != 0:
    raise RuntimeError('Select Runtime > Change runtime type > GPU before continuing')

## Experiment settings
Start with one question and `supporting` if you only want a quick pipeline test. The default below keeps distractor passages and is the more honest first experiment.

In [ ]:
MODEL_ID = 'Qwen/Qwen3.5-2B'
HOTPOT_CONFIG = 'distractor'
HOTPOT_SPLIT = 'validation'
MAX_QUESTIONS = 3
START_INDEX = 0
CONTEXT_MODE = 'all'  # 'all' or 'supporting'
PORT = 8000
DATA_DIR = '/content/hotpotqa_data'
OUTPUT_DIR = '/content/colab_outputs/hotpotqa_v1'

## Clone and install

In [ ]:
REPO_DIR = '/content/SmallScaledAutoSchemaKG'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--pre', 'vllm', '--torch-backend=auto'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio', 'torchvision'], check=False)
subprocess.run(['uv', 'pip', 'install', '--system', 'torchvision', '--torch-backend=auto'], check=True)
subprocess.run([sys.executable, '-c', "import torch, torchvision, vllm; print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'torchvision', torchvision.__version__, 'vLLM', vllm.__version__)"], check=True)

## Stream and prepare HotpotQA

In [ ]:
subprocess.run([
    sys.executable, 'scripts/prepare_hotpotqa.py',
    '--config', HOTPOT_CONFIG, '--split', HOTPOT_SPLIT,
    '--max-questions', str(MAX_QUESTIONS), '--start-index', str(START_INDEX),
    '--context-mode', CONTEXT_MODE, '--output-dir', DATA_DIR, '--overwrite'
], check=True)

In [ ]:
import json
from pathlib import Path
metadata = json.loads(Path(DATA_DIR, 'dataset_metadata.json').read_text(encoding='utf-8'))
manifest = json.loads(Path(DATA_DIR, 'qa_manifest.json').read_text(encoding='utf-8'))
print(json.dumps(metadata, indent=2))
print('First question:', json.dumps(manifest[0], indent=2, ensure_ascii=False))

## Start local vLLM server

In [ ]:
import shutil, time
import requests
LOG_PATH = '/content/qwen35_hotpotqa_vllm.log'
vllm_executable = shutil.which('vllm')
if not vllm_executable:
    raise RuntimeError('vLLM executable was not installed')
server_log = open(LOG_PATH, 'w', encoding='utf-8')
server_cmd = [vllm_executable, 'serve', MODEL_ID, '--host', '127.0.0.1', '--port', str(PORT), '--dtype', 'half', '--max-model-len', '8192', '--gpu-memory-utilization', '0.85', '--language-model-only']
server = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)
deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        server_log.flush()
        raise RuntimeError(f'vLLM stopped early. Read {LOG_PATH}')
    try:
        response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
        if response.ok:
            print('Local model server is ready')
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f'vLLM did not become ready. Read {LOG_PATH}')

## Build the HotpotQA knowledge graph

In [ ]:
subprocess.run([
    sys.executable, 'scripts/run_colab_v1.py', '--model', MODEL_ID,
    '--base-url', f'http://127.0.0.1:{PORT}/v1',
    '--data-dir', DATA_DIR, '--filename-pattern', 'hotpotqa_corpus',
    '--experiment-metadata', str(Path(DATA_DIR, 'dataset_metadata.json')),
    '--output-dir', OUTPUT_DIR, '--overwrite'
], check=True)

In [ ]:
summary = json.loads(Path(OUTPUT_DIR, 'run_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
assert summary.get('nodes', 0) > 0, 'The graph contains no nodes'
assert summary.get('edges', 0) > 0, 'The graph contains no edges'
shutil.copytree(DATA_DIR, str(Path(OUTPUT_DIR, 'provenance')), dirs_exist_ok=True)

## Package and save
The runtime filesystem is temporary. Download the ZIP or copy it to Drive before disconnecting.

In [ ]:
archive = shutil.make_archive('/content/autoschemakg_hotpotqa_v1', 'zip', OUTPUT_DIR)
print('Created:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass